In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, precision_score, recall_score, f1_score, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Reader, Dataset, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy
import random



# Graph-Based Recommendation System using PageRank and Collaborative Filtering

## Project Overview

In this notebook, I will build a recommendation system using the **MovieLens 20M dataset**, which contains 20 million movie ratings from over 138,000 users on 27,000 movies. I will implement a **hybrid recommendation approach** that combines **Collaborative Filtering** and **PageRank** to provide personalized movie recommendations.

Link to the dataset: https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset

- **Collaborative Filtering (CF)** is a popular technique in recommendation systems that relies on past behavior and preferences of users. There are two types of collaborative filtering:
  - **User-based Collaborative Filtering**: Recommending items based on the similarity between users.
  - **Item-based Collaborative Filtering**: Recommending items based on the similarity between items (in this case, movies).

- **PageRank** is an algorithm traditionally used to rank webpages based on the link structure of the internet. In this project, I will adapt PageRank to rank movies and/or users based on their interaction in the movie rating network. By applying PageRank to the user-item interaction graph, I can calculate the "importance" of movies and users, which can enhance the recommendations.

By combining **Collaborative Filtering** with **PageRank**, the system will provide better recommendations by incorporating both user-item similarities and the network structure of interactions.

## Dataset

The dataset I will use is the **MovieLens 20M dataset**, which contains the following files:

- **movies.csv**: Contains data about the movies (movieId, title, genre).
- **ratings.csv**: Contains user ratings for movies (userId, movieId, rating, timestamp).
- **tags.csv**: Contains user-generated tags for movies (userId, movieId, tag, timestamp).
- **links.csv**: Contains links to external databases like IMDb (movieId, IMDbId, tmdbId).

For this project, I will focus on the **ratings.csv** file to build the recommendation system, which contains the user-item interactions (ratings for movies by users).

## Approach

I will use a **hybrid approach** combining **Item-based Collaborative Filtering** and **PageRank** to generate movie recommendations. The steps involved are:

1. **Data Preprocessing**: 
   - I will load the **ratings.csv** file and clean the data.
   - I will create a **user-item interaction matrix** where rows represent users, columns represent movies, and values represent ratings.

2. **Similarity Calculation using Collaborative Filtering**:
   - I will compute the similarity between movies using **Cosine Similarity**.
   - I will recommend movies based on the similarity between movies the user has rated highly.

3. **PageRank Calculation on the User-Item Graph**:
   - I will create a **bipartite graph** with users and items (movies) as nodes, and the ratings as edges.
   - Using **PageRank**, I will calculate the importance of movies (and/or users) within the graph, which helps to prioritize popular and relevant items in the recommendations.

4. **Hybrid Model**:
   - For each user, I will combine the recommendations from **Collaborative Filtering** and **PageRank** to generate the final list of movie recommendations.
   - The final recommendations will be ranked based on a weighted combination of both the similarity scores from collaborative filtering and the importance scores from PageRank.

5. **Evaluation**:
   - I will evaluate the model using standard **evaluation metrics** such as **Root Mean Squared Error (RMSE)** to assess the quality of the recommendations.

## Steps in the Notebook

1. **Data Loading and Preprocessing**
2. **Building the User-Item Interaction Matrix**
3. **Collaborative Filtering: Cosine Similarity**
4. **Creating the User-Item Graph and Applying PageRank**
5. **Combining Collaborative Filtering and PageRank for Recommendations**
6. **Visualizations**
7. **Evaluating the Recommendation System**
8. **Conclusion**

---

# (1) Data Loading and Preprocessing

In [ ]:
movies = pd.read_csv("/kaggle/input/movielens-20m-dataset/movie.csv")
ratings = pd.read_csv("/kaggle/input/movielens-20m-dataset/rating.csv")
links = pd.read_csv("/kaggle/input/movielens-20m-dataset/link.csv")
tags = pd.read_csv("/kaggle/input/movielens-20m-dataset/tag.csv")
movies.shape, ratings.shape, links.shape, tags.shape

In [ ]:
movies.head(3)

In [ ]:
ratings.head(3)

In [ ]:
links.head(3)

In [ ]:
tags.head(3)

In [ ]:
ratings.info()

In [ ]:
movies.info()

In [ ]:
ratings = ratings.loc[:,["userId","movieId","rating"]]
movies = movies.loc[:,["movieId","title"]]

In [ ]:
movie_and_rating = pd.merge(movies,ratings)
movie_and_rating.shape

In [ ]:
movie_and_rating.head(3)

### Data Overview

As shown in the dataframe above, the dataset contains four key features:
- **Movie ID**: The unique identifier for each movie.
- **Title**: The name of the movie.
- **User ID**: The unique identifier for each user.
- **Rating**: The rating given by the user to the movie.

Using this dataset, I will build an **item-based recommendation system**, where the goal is to recommend movies based on the similarity between items (movies in this case) that a user has rated highly.

### Data Size and Sampling

The dataset consists of **20 million ratings**, which is quite large. Handling such a large dataset can be resource-intensive, especially local environment. Processing this much data may lead to performance issues, such as memory overflow or slow computation.

To mitigate this, I will work with a **random sample of 10 K ratings** for the purpose of building and testing the recommendation system. This smaller subset will allow for faster computation and a more manageable learning process, while still maintaining the integrity of the item-based collaborative filtering approach.


In [ ]:
movie_and_rating_sampled = movie_and_rating.sample(n=10000, random_state=1)
movie_and_rating_sampled.shape

In [ ]:
movie_and_rating_sampled.head(10)

In [ ]:
pivot_table = movie_and_rating_sampled.pivot_table(index = ["userId"],columns = ["title"],values = "rating")
pivot_table.shape

In [ ]:
pivot_table.head(10)

# (2) Building the User-Item Interaction Matrix

### Data Overview: User-Item Matrix

As can be seen from the table above, the data is structured such that:
- **Rows** represent **users**.
- **Columns** represent **movies**.
- **Values** represent the **ratings** given by users to movies.

### User-Item Matrix: 
I have created a pivot table where each row is a user and each column is a movie. The values represent the ratings given by users to movies.

### NaN Handling: 
Fill NaN values with 0, assuming the user hasn't rated the movie.

### Check Movie Correlation
After replacing the Nan with 0, I am checking the correlation of a random movie, to get an idea of the correlations. 

In [ ]:
pivot_table = pivot_table.fillna(0)

In [ ]:
pivot_table.head(10)

In [ ]:
random_movie = np.random.choice(pivot_table.columns)
movie_ratings = pivot_table[random_movie]
similarity_with_other_movies = pivot_table.corrwith(movie_ratings)
similarity_with_other_movies = similarity_with_other_movies.sort_values(ascending=False)
similarity_with_other_movies.head(5)


In [ ]:
similarity_with_other_movies.describe()

### Correlation Statistics Overview

The correlation values between movies are summarized as follows:

- **Count**: The number of movie pairs compared is **[count]**.
- **Mean**: The average correlation value is **[mean]**, indicating the general tendency of similarity between movies.
- **Standard Deviation (std)**: The standard deviation is **[std]**, reflecting how much the correlations vary around the mean.
- **Min**: The lowest observed correlation is **[min]**, indicating the strongest negative correlation between some movie pairs.
- **25th Percentile (25%)**: The 25th percentile value is **[25%]**, meaning that 25% of movie pairs have a correlation lower than this.
- **50th Percentile (50%) / Median**: The median correlation value is **[50%]**, which represents the point where half of the movie pairs show higher, and half show lower correlation.
- **75th Percentile (75%)**: The 75th percentile value is **[75%]**, meaning that 75% of movie pairs have a correlation value less than this.
- **Max**: The highest observed correlation is **[max]**, indicating a perfect positive correlation between certain movie pairs (which is the same movie in our case)

### Interpretation:

- The correlation values range from **[min]** (perfect negative correlation) to **[max]** (perfect positive correlation), with most values typically clustering around the mean value, which is **[mean]**.
- A high **standard deviation** of **[std]** suggests that there is significant variability in the similarity scores between different movie pairs.
- The data shows a mix of **positive** and **negative** correlations, with some movie pairs being highly similar (**[max]**) and others being highly dissimilar (**[min]**).

These statistics help us understand the overall relationships between movies based on user ratings, and provide insight into the diversity of similarities in the movie dataset.


# (3) Collaborative Filtering: Cosine Similarity

### Normalization: 

I am normalizing the user-item matrix to ensure that cosine similarity calculations are fair (considering different rating scales).

### Cosine Similarity: 
After normalizing, I am computing the cosine similarity between the movies, which measures how similar two movies are based on user ratings.

### Recommendation Scenario

Suppose we have a movie website, and a movie **X** has been watched and rated by a group of people. The question is: **which movie should we recommend to users who watched X?**

To answer this question, we will calculate the **similarity** between **X** and other movies. Movies that are similar to **X** will be recommended to users who rated it highly, based on the principle that users who liked one movie are likely to enjoy similar movies.

- I will randomly select a movie **X**.
- I will calculate the **similarity** between **X** and other movies.
- Use a similarity measure like **Cosine Similarity** to identify movies that are most similar to **X**.

## Cosine Similarity

**Cosine Similarity** is a metric used to measure how similar two vectors are, regardless of their magnitude. It is commonly used in text analysis and recommendation systems to compare the similarity between users, items, or documents based on their features or interactions.

### **Formula for Cosine Similarity**

The formula for cosine similarity between two vectors **A** and **B** is:

$$
\text{Cosine Similarity} = \frac{A \cdot B}{\|A\| \|B\|}
$$

Where:
- \( A \cdot B \) is the **dot product** of vectors A and B.
- \( \|A\| \) and \( \|B\| \) are the **Euclidean norms** (magnitudes) of vectors A and B, respectively.

### **Steps to Compute Cosine Similarity:**

1. **Dot Product**: Multiply the corresponding elements of the vectors A and B and sum the results.
   
   $$
   A \cdot B = \sum_{i=1}^{n} (A_i \times B_i)
   $$

2. **Euclidean Norm (Magnitude)**: Calculate the magnitude of each vector. The Euclidean norm is the square root of the sum of the squares of the elements.
   
   $$
   \|A\| = \sqrt{\sum_{i=1}^{n} A_i^2}
   $$

3. **Cosine Similarity**: Finally, divide the dot product by the product of the magnitudes of the two vectors.

$$
\text{Cosine Similarity} = \frac{A \cdot B}{\|A\| \|B\|}
$$

The result of cosine similarity will be a value between **-1** and **1**:
- A value of **1** means the vectors are identical (i.e., the angle between them is 0 degrees).
- A value of **0** means there is no similarity (i.e., the vectors are orthogonal).
- A value of **-1** means the vectors are diametrically opposed (i.e., they point in opposite directions).

### **Cosine Similarity in Recommendation Systems**

In recommendation systems, cosine similarity is often used to measure the similarity between users or items. It can be applied in **Collaborative Filtering**, where the goal is to recommend items to a user based on the preferences of similar users.

#### **Use Case Example:**
- **User-Item Interaction Matrix**: For a movie recommendation system, each row represents a user, and each column represents a movie. The entries in the matrix can be the ratings given by users to movies.
- **Cosine Similarity between Users**: By calculating the cosine similarity between users based on their ratings (or implicit feedback like clicks or views), we can identify users with similar tastes.
- **Item-Based Recommendations**: We can calculate the similarity between items (e.g., movies) and recommend items that are similar to those a user has rated highly.

### **Advantages of Cosine Similarity:**
1. **Normalization**: It is robust to the magnitude of the vectors, so it can be used in cases where the absolute ratings vary significantly (e.g., some users might give higher ratings than others).
2. **Interpretability**: The result is easy to interpret, as it ranges between -1 and 1.

### **Disadvantages of Cosine Similarity:**
1. **Sparsity**: In a typical recommendation system, user-item interaction matrices are sparse (many users haven’t rated many items). This can lead to difficulties in calculating meaningful similarities when data is missing.
2. **Context Ignorance**: Cosine similarity only looks at the direction of vectors (i.e., the patterns of ratings), and doesn’t account for external factors like the context of ratings.


----


In [ ]:
# Normalize the data for better similarity computation
normalized_data = normalize(pivot_table, axis=0)
cosine_sim = cosine_similarity(normalized_data.T)

# Create a DataFrame for the similarity matrix
cosine_sim_df = pd.DataFrame(cosine_sim, index=pivot_table.columns, columns=pivot_table.columns)

In [ ]:
cosine_sim_df.head(10)

# (4) Creating the User-Item Graph and Applying PageRank

### Bipartite Graph: 
Here, I am creatimg a bipartite graph where users and movies are nodes, and edges represent user-item interactions based on the ratings.

### PageRank: 
Post creating the graph, I am applying the PageRank algorithm to rank the movies based on their importance in the interaction network. The result is a list of movies ordered by their PageRank scores.

In [ ]:
# Create a bipartite graph of user-item interactions (users and movies as nodes)
G = nx.Graph()

G.add_nodes_from(pivot_table.index, bipartite=0)
G.add_nodes_from(pivot_table.columns, bipartite=1)

print(len(pivot_table.index))
print(len(pivot_table.columns))

In [ ]:
# Adding edges
for user in pivot_table.index:
    for movie in pivot_table.columns:
        rating = pivot_table.loc[user, movie]
        if rating > 0:  # Add edges only if the user has rated the movie
            G.add_edge(user, movie, weight=rating)


In [ ]:
# Plot the graph
# plt.figure(figsize=(12, 12))

# pos = nx.spring_layout(G, k=0.15, seed=42)

# nx.draw(G, pos, with_labels=True, node_size=50, font_size=8, node_color='skyblue', edge_color='gray', alpha=0.7)

# # Display the plot
# plt.title("User-Item Interaction Graph")
# plt.show()

### I tried to build the entire node graph, but it is very haphazard given the high value of number of nodes, so I'm generating a graph using subset of the data below

In [ ]:
list(G.edges)[:10]

In [ ]:
edges = list(G.edges)

if not edges:
    print("The graph has no edges.")
else:
    selected_edges = []
    selected_nodes = set()

    for edge in edges:
        if len(selected_nodes) < 30:
            u, v = edge
            if u not in selected_nodes or v not in selected_nodes:
                selected_edges.append(edge)
                selected_nodes.update([u, v])

    #Create a subgraph from the selected edges
    subgraph = G.edge_subgraph(selected_edges).copy()

    pos = nx.circular_layout(subgraph)

    # Plot the subgraph
    plt.figure(figsize=(12, 12))
    
    node_colors = ['lightblue' if isinstance(node, int) else 'lightgreen' for node in subgraph.nodes]

    # Draw the graph without labels
    nx.draw(subgraph, pos, with_labels=False, node_size=800, font_size=10,
            node_color=node_colors, edge_color='gray', width=1.5, alpha=0.8)

   # Users (first element in the tuple) will have user ID as label
    user_labels = {node: str(node) for node in subgraph.nodes if isinstance(node, int)}

    # Movies (second element in the tuple) will have movie title as label
    movie_labels = {node: node for node in subgraph.nodes if isinstance(node, str)}

    movie_pos = {node: (pos[node][0] + 0.1, pos[node][1]) for node in movie_labels}
    nx.draw_networkx_labels(subgraph, movie_pos, labels=movie_labels, font_size=10, font_color='black')
    nx.draw_networkx_labels(subgraph, pos, labels=user_labels, font_size=10, font_color='black')

    plt.title(f"Circular User-Movie Interaction Subgraph ({len(subgraph.nodes)} Nodes, {len(subgraph.edges)} Edges)")
    plt.show()


## Visualization of User-Item Interaction Subgraph

In the above section, I have created a visualization of the bipartite graph representing user-movie interactions. The graph is constructed using NetworkX, where the **users** are represented as integer IDs and the **movies** are represented by their titles (strings). 

## Steps Involved:
1. **Graph Construction**:
   - I am extracting the edges from the original bipartite graph. Each edge in the graph connects a user to a movie, where the first element of the edge is the user ID and the second element is the movie title.

2. **Node Selection**:
   - To keep the visualization manageable, i am selecting a subset of 20 nodes. The nodes include users and movies that are part of the selected edges. The selected edges are those where either the user or the movie is not already in the selected set of nodes, ensuring that the nodes are connected.

3. **Circular Layout**:
   - The selected subgraph is visualized using a **circular layout**, which places the nodes evenly in a circular formation.

4. **Node Coloring**:
   - Nodes representing **users** are colored **lightblue**, and nodes representing **movies** are colored **lightgreen**.

5. **Labeling**:
   - Each **user node** is labeled with its **user ID**, and each **movie node** is labeled with its **movie title**.
   - To avoid overlap, movie labels are offset slightly to the right of their corresponding movie nodes.

6. **Graph Visualization**:
   - The final graph is displayed with edges connecting the selected users to the selected movies, providing a clear view of the bipartite structure.

## Visualization Output:
- The plot will show a bipartite graph where **users** and **movies** are shown as nodes.
- The edges between users and movies indicate interactions, and the circular layout provides a clean, visually appealing representation.

---


## Ranking Movies Using PageRank

In this section, I apply the **PageRank** algorithm to my bipartite graph, which consists of users and movies. PageRank is a graph-based algorithm that ranks the nodes based on their importance in the network. In our case it is the movies. Essentially, it helps to identify movies that are more "central" or influential because they have more connections or interactions from users.

First, I calculate the PageRank scores for all nodes in the graph using the **`networkx.pagerank`** function. After that, I filter out the movie nodes and extract their PageRank scores. These scores tell me how important each movie is in terms of user interactions, considering both direct and indirect connections.

Next, I sort the movies by their PageRank scores in descending order. This way, I can find the top-ranked movies based on their influence in the network. Finally, I display the top 10 movies with the highest PageRank scores.

By using PageRank, I can better understand which movies are more importan in the user-item graph, which can be helpful for making recommendations.


In [ ]:
# Apply PageRank to the graph to rank movies (we focus on movie nodes)
pagerank_scores = nx.pagerank(G, alpha=0.85)

# Extract the PageRank scores for movies
movie_pagerank_scores = {movie: score for movie, score in pagerank_scores.items() if movie in pivot_table.columns}

# Sort the movies based on their PageRank scores
sorted_movies_by_pagerank = sorted(movie_pagerank_scores.items(), key=lambda x: x[1], reverse=True)

# Display the top 10 movies based on PageRank
sorted_movies_by_pagerank[:10]

# (5). Combining Collaborative Filtering and PageRank for Recommendations

### Hybrid Recommendations: 
Here, I am implementing the function get_recommendations() which combines the results of cosine similarity and PageRank. Movies that are similar to those rated by the user (based on collaborative filtering) are ranked according to both similarity and their PageRank scores.

### Output:
The final output returned is the top 10 recommended movies for a user. 

In [ ]:

print(pivot_table.index)


In [ ]:
def get_recommendations(user_id, G, pivot_table, top_n=10, c_f_importance=0.5):
    """
    Get movie recommendations for a user based on a hybrid approach combining collaborative filtering 
    (cosine similarity) and PageRank.
    
    Parameters:
    - user_id: The user for whom we are generating recommendations.
    - G: The bipartite graph (user-item interaction graph).
    - pivot_table: The user-item matrix containing ratings.
    - top_n: Number of recommendations to return.
    
    Returns:
    - top_n_recommendations: List of top 'n' movie recommendations.
    """
    
    train_data = pivot_table.copy()
    train_data.fillna(0, inplace=True)

    user_ratings = train_data.loc[user_id].values
    rated_movies = train_data.columns[train_data.loc[user_id] > 0]

    # Generate cosine similarity between the user and all other users based on ratings
    other_users_ratings = train_data.drop(user_id, axis=0).values
    similarities = cosine_similarity([user_ratings], other_users_ratings)

    # Get the most similar users
    similar_users_idx = similarities.argsort()[0, ::-1]

    recommended_movies = set()
    for user_idx in similar_users_idx:
        similar_user_id = train_data.index[user_idx]
        similar_user_ratings = train_data.loc[similar_user_id]
        
        new_movies = similar_user_ratings[similar_user_ratings > 0].index.difference(rated_movies)
        recommended_movies.update(new_movies)
        
        if len(recommended_movies) >= top_n:
            break
    
    # Apply PageRank on the movie nodes (bipartite graph)
    movie_nodes = [node for node in G.nodes if isinstance(node, str)]
    subgraph = G.subgraph(movie_nodes)
    pagerank_scores = nx.pagerank(subgraph)
    
    movie_scores = {}
    for movie in recommended_movies:
        cf_score = similarities[0, similarities.argsort()[0, ::-1]][0]
        pr_score = pagerank_scores.get(movie, 0)
        
        combined_score = (c_f_importance) * cf_score + (1 - c_f_importance) * pr_score
        movie_scores[movie] = combined_score
    
    top_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_n_recommendations = [movie for movie, score in top_movies]
    
    return top_n_recommendations

In [ ]:
# Usage
evaluation_results = []
for i in range(0,100):
    user_id = pivot_table.index[i]
    recommended_movies = get_recommendations(user_id, G, pivot_table, top_n=10)
    evaluation_results.append({
            'User ID': user_id,
            'Recommendation' : recommended_movies
        })

# TODO: REMOVE THIS SECTION

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df.head(10)


# (6) Visualizations

### PageRank Distribution: 
In this section, I am visualizing the distribution of PageRank scores across movies, showing how many movies are highly ranked versus less ranked.

### Cosine Similarity Distribution: 
I am also visualizing the distribution of cosine similarity values between movies to understand how similar most movies are.

In [ ]:
# Plot the distribution of PageRank scores for movies
pagerank_values = list(movie_pagerank_scores.values())
plt.hist(pagerank_values, bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of PageRank Scores for Movies')
plt.xlabel('PageRank Score')
plt.ylabel('Number of Movies')
plt.show()

# Plot the distribution of cosine similarity scores between movies
cosine_values = cosine_sim_df.values.flatten()
plt.hist(cosine_values, bins=50, color='lightgreen', edgecolor='black')
plt.title('Distribution of Cosine Similarity Scores')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()


# (7) Evaluating the Recommendation System

### Introduction

Evaluating the performance of a recommendation system is essential to understand how effectively it provides relevant suggestions to users. In this project, I have implemented a **hybrid recommendation system** that combines **PageRank** and **Collaborative Filtering** techniques to generate personalized recommendations for users. 

To assess the quality of the recommendations, I will evaluate the system using common evaluation metrics, such as **Precision**, **Recall**, **F1-Score**. These metrics will allow us to measure both the relevance of the recommendations (i.e., whether the recommended items are actually liked by the user) and the accuracy of the predicted ratings (i.e., how close the predicted ratings are to the actual ratings given by users).

- **Precision** measures how many of the recommended items are relevant.
- **Recall** measures how many of the relevant items are included in the recommendations.
- **F1-Score** provides a balance between precision and recall.
By evaluating these metrics, I can determine the overall effectiveness of the recommendation system in generating relevant and accurate suggestions.

### Evaluation Metrics

To evaluate the system, I selected a set of users and evaluated the top recommended movies for each user. The evaluation procedure included:

1. **Precision**: The proportion of recommended movies that the user has rated highly (i.e., relevant recommendations).
2. **Recall**: The proportion of the user’s rated movies that appear in the top 10 recommendations (i.e., how well the system captures the user's preferences).
3. **F1-Score**: The harmonic mean of precision and recall, giving a single metric that balances both.

Here, I am going to evaluate my model by the below approach:

### Leave out recommendation: 
Leaving out a subset of data for deriving the similarity and then checking if the user's highest rated movies appear in our recommendation.

---


In [ ]:
pivot_table.head(3)

## Leave out recommendation: 
### **Challenges in Evaluating Recommendation Systems Without Ground Truth**

Evaluating a recommendation system can be tricky, especially when we don't have a **ground truth** to compare our recommendations against. Ground truth refers to knowing exactly which recommendations are correct for a user—something we usually don't have in real-world systems. Without this, it's difficult to say with certainty whether our system is recommending the right items.

In our case, since we don't have ground truth data (like knowing exactly which movies a user would like), we are relying on indirect methods for evaluation. This makes the results less clear and harder to interpret.

### **Leaving Out High-Rated Movies: Why It Affects the Results**

To evaluate our system, we decided to **leave out the highest-rated movies** by each user. The idea is to test how well the system can recommend movies that the user hasn’t rated yet but might enjoy. This simulates real-world situations where we don’t have complete information about a user’s preferences.

However, leaving out highly rated movies introduces a few problems:

#### 1. **Bias in Recommendations**
When we remove the highest-rated movies, the system doesn’t have access to the **most important preferences** of the user. These high-rated movies are typically the ones that best represent the user’s true interests. Without them, the system might not make the most accurate recommendations, leading to lower quality results.

#### 2. **Problems with Similarity**
Our system relies on **similarity measures** (like cosine similarity) to find movies that are similar to the ones a user has liked before. If we leave out the top-rated movies, the system has less information to work with, which can make it harder to find truly relevant movies. This means the recommendations might not be as good.

#### 3. **Skewed Evaluation Metrics**
Because we’re leaving out the most important ratings, the evaluation metrics (like precision, recall, and F1-score) might not give an accurate picture of the system’s performance. The system might fail to recommend the best movies simply because it doesn’t have enough information about the user’s true preferences.

---

### **Why the Results May Not Be As Good as Expected**

Since we’re simulating a situation where the system doesn’t have full access to user ratings, the evaluation results might not be perfect. Here’s why:

- **Less Data**: By leaving out the highest-rated movies, we’re reducing the amount of **valuable information** the system can use to make recommendations. With less data, the system can’t make the best choices.
  
- **Weak Similarity Measures**: Without the top-rated movies, the similarity calculation becomes **less accurate**, which means the system might suggest movies that aren’t as relevant.

- **No Clear Benchmark**: Without a ground truth to compare our recommendations against, it’s hard to know exactly how well the system is performing.

---

### **Conclusion: What This Means for Our Results**

Even though our system might not perform as well as we expected under these conditions, this type of evaluation helps us understand how the model works when it has incomplete information. 

While the evaluation metrics (like precision, recall, and F1-score) might not look great, they still give us useful insights into how the system behaves when crucial data is missing. For more accurate results, we’d need a **richer set of data** or **user feedback** to act as ground truth.

In summary, while the system may not perform at its best with missing data, this evaluation still helps us identify areas where the system could be improved. With more complete information, we could expect better results.

In [ ]:

def leave_out_movies(pivot_table, user_id, leave_out_count):
    """
    Leave out the top-rated movies for the user based on their ratings.
    
    Args:
    - pivot_table: The user-item matrix containing ratings.
    - user_id: The user whose movies are being left out.
    - leave_out_count: The number of top-rated movies to leave out.
    
    Returns:
    - train_pivot_table: The modified pivot table with the top-rated movies excluded.
    - left_out_movies: The list of movies that were left out (highest-rated).
    """
    # Step 1: Get the user's ratings
    user_ratings = pivot_table.loc[user_id]
    
    # Step 2: Sort the movies by ratings (highest first)
    top_rated_movies = user_ratings.sort_values(ascending=False).head(leave_out_count)
    
    # Step 3: Leave out the top-rated movies by setting their ratings to NaN
    left_out_movies = top_rated_movies.index.tolist()  # Movies that are left out
    
    train_pivot_table = pivot_table.copy()
    train_pivot_table.loc[user_id, left_out_movies] = np.nan  # Remove the top-rated movies
    
    return train_pivot_table, left_out_movies


In [ ]:
def evaluate_leave_out(user_id, pivot_table, G, leave_out_count=1, top_n=10):
    """
    Evaluate the recommendation system for a given user by leaving out 'leave_out_count' movies, 
    generating recommendations, and comparing with the ground truth (user ratings).
    
    Args:
    - user_id: The user for whom we are generating recommendations.
    - pivot_table: The user-item matrix containing ratings.
    - G: The bipartite graph for collaborative filtering.
    - leave_out_count: Number of movies to leave out for the user.
    - top_n: Number of recommendations to generate.
    
    Returns:
    - evaluation_results: A dictionary of evaluation metrics.
    """
    
    # Step 1: Leave out 'leave_out_count' movies for the user
    train_pivot_table, left_out_movies = leave_out_movies(pivot_table, user_id, leave_out_count)
    
    # Step 2: Generate recommendations for the user based on the remaining ratings
    recommended_movies = get_recommendations(user_id, G, train_pivot_table, top_n=top_n)
    
    # Step 3: Get the ground truth for the user (relevant items)
    relevant_items = pivot_table.loc[user_id].dropna()  # Ground truth (rated movies)
    relevant_ratings = relevant_items[relevant_items > 0]  # Keep only rated items
    
    # Step 4: Calculate Precision, Recall, F1-Score (as previously discussed)
    y_pred = [1] * leave_out_count
    y_true = [1 if movie in recommended_movies else 0 for movie in left_out_movies]
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Step 5: Return the evaluation metrics
    evaluation_results = {
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }
    
    return evaluation_results

In [ ]:

def evaluate_leave_out_multiple(user_ids, pivot_table, G, leave_out_count=2,top_n=10):
    """
    Evaluate the recommendation system for multiple users and return the evaluation metrics as a DataFrame.
    
    Args:
    - user_ids (list): List of user IDs to evaluate.
    - pivot_table (pd.DataFrame): The pivot table containing user-item interactions (ratings).
    - model (surprise.AlgoBase): The collaborative filtering model to generate recommendations.
    - benchmark_model (function): Function to generate benchmark recommended movies (e.g., `get_recommendation_benchmark`).
    - G (networkx.Graph): The bipartite graph used for generating recommendations.
    - top_n (int): The number of top recommendations to consider (default 10).
    
    Returns:
    - pd.DataFrame: DataFrame containing the evaluation metrics for each user.
    """
    evaluation_results = []

    for user_id in user_ids:
        
        evaluation_metrics = evaluate_leave_out(user_id, pivot_table, G, leave_out_count=leave_out_count, top_n=top_n)
        
        # Add the evaluation metrics to the results
        evaluation_results.append({
            'User ID': user_id,
            'Precision': evaluation_metrics['Precision'],
            'Recall': evaluation_metrics['Recall'],
            'F1-Score': evaluation_metrics['F1-Score']
        })
    
    evaluation_df = pd.DataFrame(evaluation_results)
    return evaluation_df


In [ ]:
multiple_evaluations = []

In [ ]:
user_ids = pivot_table.index[:100]
evaluation_df = evaluate_leave_out_multiple(user_ids, pivot_table, G, leave_out_count=2,top_n=2)
multiple_evaluations.append({
            'Leave Out Count': 2,
            'Top N': 2,
            'Precision': evaluation_df['Precision'].mean(),
            'Recall': evaluation_df['Recall'].mean(),
            'F1-Score': evaluation_df['F1-Score'].mean()
        })
evaluation_df['Precision'].mean(), evaluation_df['Recall'].mean(), evaluation_df['F1-Score'].mean()


In [ ]:
evaluation_df = evaluate_leave_out_multiple(user_ids, pivot_table, G, leave_out_count=2,top_n=100)
multiple_evaluations.append({
            'Leave Out Count': 2,
            'Top N': 100,
            'Precision': evaluation_df['Precision'].mean(),
            'Recall': evaluation_df['Recall'].mean(),
            'F1-Score': evaluation_df['F1-Score'].mean()
        })
evaluation_df['Precision'].mean(), evaluation_df['Recall'].mean(), evaluation_df['F1-Score'].mean()


In [ ]:
evaluation_df = evaluate_leave_out_multiple(user_ids, pivot_table, G, leave_out_count=3,top_n=500)
multiple_evaluations.append({
            'Leave Out Count': 3,
            'Top N': 500,
            'Precision': evaluation_df['Precision'].mean(),
            'Recall': evaluation_df['Recall'].mean(),
            'F1-Score': evaluation_df['F1-Score'].mean()
        })
evaluation_df['Precision'].mean(), evaluation_df['Recall'].mean(), evaluation_df['F1-Score'].mean()


In [ ]:
evaluation_df = evaluate_leave_out_multiple(user_ids, pivot_table, G, leave_out_count=2,top_n=1500)
multiple_evaluations.append({
            'Leave Out Count': 2,
            'Top N': 1500,
            'Precision': evaluation_df['Precision'].mean(),
            'Recall': evaluation_df['Recall'].mean(),
            'F1-Score': evaluation_df['F1-Score'].mean()
        })
evaluation_df['Precision'].mean(), evaluation_df['Recall'].mean(), evaluation_df['F1-Score'].mean()


In [ ]:
evaluation_results_df = pd.DataFrame(multiple_evaluations)
evaluation_results_df

In [ ]:
print(evaluation_df)

# Evaluation of Recommendation System

## Metrics:
1. **Precision**: Measures how many of the recommended movies are relevant (out of all the movies recommended).
2. **Recall**: Measures how many of the relevant movies are recommended (out of all the movies the user actually rated).
3. **F1-Score**: A harmonic mean of Precision and Recall. It balances Precision and Recall, and is a good metric when you need a single measure of recommendation performance.

---

## Observations:

### 1. (leave_out_count = 2, top_n = 2)
- **Precision**: **0.0**
- **Recall**: **0.0**
- **F1-Score**: **0.0**

**Interpretation**: With only **2 recommendations** and **leave_out_count = 2**, the system appears to be **failing entirely** to recommend any relevant movies, resulting in **Precision = 0** (0% relevant recommendations) and **Recall = 0** (0% of relevant movies recommended). The F1-Score is also **0**, indicating a poor recommendation performance with this setup. This suggests that with very few recommendations (`top_n = 2`), the system is not finding any relevant items.


### 2. (leave_out_count = 2, top_n = 100)
- **Precision**: **0.435**
- **Recall**: **0.8**
- **F1-Score**: **0.557**

**Interpretation**: With a smaller number of recommendations (**100**), the Precision drops to **0.435**, which means **43.5%** of the recommended movies are relevant. However, the Recall is still relatively high at **0.8** (**80%**), indicating that the system is still able to recommend **80%** of the relevant items. 

### 3. (leave_out_count = 3, top_n = 500)
- **Precision**: **0.407**
- **Recall**: **0.91**
- **F1-Score**: **0.547**

**Interpretation**: With **leave_out_count = 3**, the Precision is even lower (**0.407**), but Recall increases significantly to **0.91**. This indicates that with more movies left out, the system is able to capture **91%** of the relevant movies, but is also recommending a higher proportion of irrelevant movies (Precision drops). The F1-Score is a little lower (**0.547**) than before.

### 4. (leave_out_count = 2, top_n = 1500)
- **Precision**: **0.7**
- **Recall**: **0.95**
- **F1-Score**: **0.783**

**Interpretation**: With **1500 recommendations**, Precision increases further to **0.7**, meaning **70%** of the recommended movies are relevant. Recall increases even more to **0.95**, meaning the system is recommending **95%** of the relevant movies. This combination results in a higher **F1-Score** (**0.783**), showing that the system is becoming more accurate and balanced as we increase the number of recommendations.

---

## General Observations:

### **As `top_n` increases** (i.e., as we recommend more movies):
- **Precision** tends to increase (because there are more opportunities to include relevant items in the recommendations).
- **Recall** also tends to increase (because the system is recommending a larger portion of the relevant items).
- **F1-Score** improves as a result, reflecting a better balance between Precision and Recall.



## Visualizing the Evaluation metrics

In [ ]:
def plot_line_metrics(df):
    plt.figure(figsize=(12, 6))

    # Plot Precision, Recall, F1-Score
    plt.plot(df['User ID'], df['Precision'], label='Precision', marker='o', color='blue')
    plt.plot(df['User ID'], df['Recall'], label='Recall', marker='o', color='green')
    plt.plot(df['User ID'], df['F1-Score'], label='F1-Score', marker='o', color='orange')

    # Add labels and title
    plt.title('Precision, Recall, and F1-Score per User ID for leave_out_count 2 and top_n as 1500')
    plt.xlabel('User ID')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    # Show the plot
    plt.show()

In [ ]:
plot_line_metrics(evaluation_df)

In [ ]:
def plot_individual_histograms(df):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot histogram for Precision
    axes[0].hist(df['Precision'], alpha=0.7, color='blue', edgecolor='black')
    axes[0].set_title('Histogram of Precision')
    axes[0].set_xlabel('Precision')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True)
    
    # Plot histogram for Recall
    axes[1].hist(df['Recall'], bins=10, alpha=0.7, color='green', edgecolor='black')
    axes[1].set_title('Histogram of Recall')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True)
    
    # Plot histogram for F1-Score
    axes[2].hist(df['F1-Score'], bins=10, alpha=0.7, color='orange', edgecolor='black')
    axes[2].set_title('Histogram of F1-Score')
    axes[2].set_xlabel('F1-Score')
    axes[2].set_ylabel('Frequency')
    axes[2].grid(True)
    
    # Adjust layout to make it look nice
    plt.tight_layout()
    plt.show()

In [ ]:
plot_individual_histograms(evaluation_df)

# Conclusion

In this project, I built a hybrid recommendation system that combines **Collaborative Filtering** (using **cosine similarity**) and **PageRank** to generate personalized movie recommendations. 

Key Points:
- The **Collaborative Filtering** approach uses user ratings to find similar movies and generate recommendations.
- **PageRank** is applied to the user-item interaction graph to rank movies based on their importance in the network.
- By combining both methods, the recommendation system can leverage both content-based and graph-based insights for better personalized recommendations.

Future Improvements:
- Tune the weighting of cosine similarity and PageRank scores for better results.
- Experiment with additional features like movie genres or temporal aspects of ratings.
- Explore more advanced models, such as **Matrix Factorization** or **Deep Learning-based Recommenders**.

This notebook demonstrated the power of combining traditional collaborative
